# Fine Tuning FLAN T5 on DialogSum with LoRA



In [1]:
SID4 = 2046
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

print("SID4:", SID4)
print("SEED:", SEED)
print("SLICE:", SLICE)
print("HP_ID:", HP_ID)
print("CLS_A:", CLS_A)
print("CLS_B:", CLS_B)


SID4: 2046
SEED: 2046
SLICE: 46
HP_ID: 0
CLS_A: 6
CLS_B: 3


## Environment Setup

Run the installation cell once in a fresh environment. Restart the runtime afterward, then run from the top while skipping the installation cell.


In [2]:
%pip uninstall -y torchao
%pip install -q --upgrade "transformers==4.56.2" "peft==0.17.1" "accelerate>=1.2,<2" "datasets>=3.0,<5" sentencepiece evaluate rouge_score


  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
import os
import random
import json
import time
import platform
import importlib.util
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, TaskType, get_peft_model
import transformers
import peft

assert importlib.util.find_spec("torchao") is None, (
    "torchao is still installed. Restart the runtime after the setup cell."
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("torchao installed:", importlib.util.find_spec("torchao") is not None)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Environment ready.")


Python: 3.13.15
PyTorch: 2.11.0+cu128
Transformers: 4.56.2
PEFT: 0.17.1
torchao installed: False
Device: cuda
GPU: Tesla T4
Environment ready.


## Task 1: Data Preparation

In [4]:
dataset = load_dataset("knkarthick/dialogsum")

print(dataset)
print()
print("Train size:", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))
print("Test size:", len(dataset["test"]))
print("Columns:", dataset["train"].column_names)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

Train size: 12460
Validation size: 500
Test size: 1500
Columns: ['id', 'dialogue', 'summary', 'topic']


In [5]:
TRAIN_SIZE = 1000
VAL_SIZE = 100

train_start = SLICE
train_end = min(train_start + TRAIN_SIZE, len(dataset["train"]))

val_start = SLICE
val_end = min(val_start + VAL_SIZE, len(dataset["validation"]))

train_dataset = dataset["train"].select(range(train_start, train_end))
val_dataset = dataset["validation"].select(range(val_start, val_end))

print("SLICE:", SLICE)
print("Training subset size:", len(train_dataset))
print("Validation subset size:", len(val_dataset))


SLICE: 46
Training subset size: 1000
Validation subset size: 100


In [6]:
def format_sample(example):
    return {
        "input": "Summarize the following dialogue:\n\n" + example["dialogue"],
        "target": example["summary"],
    }

processed_samples = train_dataset.map(format_sample)

for i in range(2):
    print("=" * 80)
    print("PROCESSED SAMPLE", i + 1)
    print()
    print("INPUT")
    print(processed_samples[i]["input"])
    print()
    print("TARGET")
    print(processed_samples[i]["target"])
    print()


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

PROCESSED SAMPLE 1

INPUT
Summarize the following dialogue:

#Person1#: Mister Ewing said we should show up at the conference center at 4 o'clock, right?
#Person2#: Yes, he specially asked us not to be late. Some of the people from our East York branch office are coming and he wants to make a good impression on them. How are you getting there?
#Person1#: I was thinking of taking my car but I think I'm just going to take the underground because there is construction on the highway. What about you?
#Person2#: I'll be taking the underground as well. Why don't we go together? I've been to the conference center only once, and I'm not sure if I can find my way around there.

TARGET
To get to the conference centre early, #Person1# and #Person2# decide to take the underground, and #Person2# proposes to go there together.

PROCESSED SAMPLE 2

INPUT
Summarize the following dialogue:

#Person1#: What's the accommodation like in London Ahmed?
#Person2#: Well, it's a bit difficult. There are thousa

## Tokenization

In [7]:
MODEL_NAME = "google/flan-t5-small"
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded:", MODEL_NAME)


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: google/flan-t5-small


In [8]:
def preprocess_function(examples):
    prompts = [
        "Summarize the following dialogue:\n\n" + dialogue
        for dialogue in examples["dialogue"]
    ]

    model_inputs = tokenizer(
        prompts,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=examples["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_val = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names,
)

tokenized_train.set_format("torch")
tokenized_val.set_format("torch")

print(tokenized_train)
print(tokenized_val)


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})


In [9]:
sample = tokenized_train[0]

decoded_input = tokenizer.decode(
    sample["input_ids"],
    skip_special_tokens=True,
)

decoded_target = tokenizer.decode(
    sample["labels"],
    skip_special_tokens=True,
)

print("Input token count:", len(sample["input_ids"]))
print("Label token count:", len(sample["labels"]))
print()
print("Decoded input:")
print(decoded_input)
print()
print("Decoded target:")
print(decoded_target)

assert decoded_input.strip() != ""
assert decoded_target.strip() != ""

print()
print("Tokenization sanity check passed.")


Input token count: 173
Label token count: 39

Decoded input:
Summarize the following dialogue: #Person1#: Mister Ewing said we should show up at the conference center at 4 o'clock, right? #Person2#: Yes, he specially asked us not to be late. Some of the people from our East York branch office are coming and he wants to make a good impression on them. How are you getting there? #Person1#: I was thinking of taking my car but I think I'm just going to take the underground because there is construction on the highway. What about you? #Person2#: I'll be taking the underground as well. Why don't we go together? I've been to the conference center only once, and I'm not sure if I can find my way around there.

Decoded target:
To get to the conference centre early, #Person1# and #Person2# decide to take the underground, and #Person2# proposes to go there together.

Tokenization sanity check passed.


## Task 2: Baseline Inference

In [10]:
test_dataset = dataset["test"]
eval_indices = [0, 1]

evaluation_examples = [
    {
        "dialogue": test_dataset[i]["dialogue"],
        "reference": test_dataset[i]["summary"],
    }
    for i in eval_indices
]

for i, example in enumerate(evaluation_examples, 1):
    print("=" * 80)
    print("EVALUATION EXAMPLE", i)
    print()
    print("DIALOGUE")
    print(example["dialogue"])
    print()
    print("REFERENCE SUMMARY")
    print(example["reference"])
    print()


EVALUATION EXAMPLE 1

DIALOGUE
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to communicate with their clients.
#Person1#: They will just have to change their communication methods. I don't want any - one using Instant Messaging in this office. It wastes too much time! Now, pl

In [11]:
def generate_summary(model, dialogue):
    prompt = "Summarize the following dialogue:\n\n" + dialogue

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    ).to(device)

    model.eval()

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            min_new_tokens=5,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True,
    ).strip()


baseline_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

baseline_outputs = [
    generate_summary(baseline_model, example["dialogue"])
    for example in evaluation_examples
]

for i in range(2):
    print("=" * 80)
    print("EXAMPLE", i + 1)
    print()
    print("BASELINE OUTPUT")
    print(baseline_outputs[i])
    print()
    print("REFERENCE")
    print(evaluation_examples[i]["reference"])
    print()

assert all(x.strip() for x in baseline_outputs)
print("Baseline generation sanity check passed.")


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

EXAMPLE 1

BASELINE OUTPUT
#Person2#: Thank you, sir.

REFERENCE
Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.

EXAMPLE 2

BASELINE OUTPUT
#Person2#: Thank you, sir.

REFERENCE
In order to prevent employees from wasting time on Instant Message programs, #Person1# decides to terminate the use of those programs and asks Ms. Dawson to send out a memo to all employees by the afternoon.

Baseline generation sanity check passed.


In [12]:
os.makedirs("outputs", exist_ok=True)

with open("outputs/baseline_outputs.json", "w", encoding="utf8") as f:
    json.dump(
        [
            {
                "example": i + 1,
                "dialogue": evaluation_examples[i]["dialogue"],
                "reference": evaluation_examples[i]["reference"],
                "baseline": baseline_outputs[i],
            }
            for i in range(2)
        ],
        f,
        indent=2,
        ensure_ascii=False,
    )

del baseline_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Baseline outputs saved.")


Baseline outputs saved.


## Task 3: Apply LoRA

In [13]:
LORA_RANK = 4
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q", "v"],
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model = get_peft_model(base_model, lora_config)
model = model.to(device)
model.config.use_cache = False

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("LoRA rank:", LORA_RANK)
print("LoRA alpha:", LORA_ALPHA)
print("LoRA dropout:", LORA_DROPOUT)
print("Target modules: q, v")
print()
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.4f}%")
print()

model.print_trainable_parameters()


LoRA rank: 4
LoRA alpha: 16
LoRA dropout: 0.05
Target modules: q, v

Total parameters: 77,133,184
Trainable parameters: 172,032
Trainable percentage: 0.2230%

trainable params: 172,032 || all params: 77,133,184 || trainable%: 0.2230


## Task 4: Fine Tuning with a Manual PyTorch Loop

In [14]:
BATCH_SIZE = 4
NUM_EPOCHS = 2
LEARNING_RATE = 2e-4
MAX_GRAD_NORM = 1.0

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
)

train_loader = DataLoader(
    tokenized_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    generator=torch.Generator().manual_seed(SEED),
)

val_loader = DataLoader(
    tokenized_val,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
)

total_training_steps = NUM_EPOCHS * len(train_loader)
warmup_steps = max(1, int(0.05 * total_training_steps))

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

print("Training batches per epoch:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Total optimizer steps:", total_training_steps)
print("Warmup steps:", warmup_steps)


Training batches per epoch: 250
Validation batches: 25
Total optimizer steps: 500
Warmup steps: 25


In [15]:
# Final pretraining sanity check

model.train()
model.zero_grad(set_to_none=True)

sanity_batch = next(iter(train_loader))
sanity_batch = {
    key: value.to(device)
    for key, value in sanity_batch.items()
}

sanity_output = model(**sanity_batch)
sanity_loss = sanity_output.loss

print("Pretraining batch loss:", float(sanity_loss))
print("Loss finite:", bool(torch.isfinite(sanity_loss)))

sanity_loss.backward()

bad_gradients = []

for name, parameter in model.named_parameters():
    if parameter.requires_grad and parameter.grad is not None:
        if not torch.isfinite(parameter.grad).all():
            bad_gradients.append(name)

print("Nonfinite gradient tensors:", len(bad_gradients))

assert torch.isfinite(sanity_loss)
assert float(sanity_loss) > 0
assert len(bad_gradients) == 0

model.zero_grad(set_to_none=True)

print("Training sanity check passed.")


/usr/local/lib/python3.13/dist-packages/transformers/data/data_collator.py:741: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
/tmp/ipykernel_5007/3215578784.py:15: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("Pretraining batch loss:", float(sanity_loss))


Pretraining batch loss: 2.606649160385132
Loss finite: True
Nonfinite gradient tensors: 0
Training sanity check passed.


In [16]:
training_history = []
global_step = 0
training_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_losses = []

    for batch_index, batch in enumerate(train_loader, start=1):
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        outputs = model(**batch)
        loss = outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Nonfinite loss at epoch {epoch}, batch {batch_index}: {loss.item()}"
            )

        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM,
        )

        if not torch.isfinite(grad_norm):
            raise RuntimeError(
                f"Nonfinite gradient norm at epoch {epoch}, batch {batch_index}"
            )

        optimizer.step()
        scheduler.step()

        global_step += 1
        loss_value = float(loss.detach().cpu())
        epoch_losses.append(loss_value)

        if global_step == 1 or global_step % 25 == 0:
            current_lr = scheduler.get_last_lr()[0]

            print(
                f"epoch={epoch} "
                f"step={global_step}/{total_training_steps} "
                f"loss={loss_value:.4f} "
                f"grad_norm={float(grad_norm):.4f} "
                f"lr={current_lr:.8f}"
            )

    train_loss = float(np.mean(epoch_losses))

    model.eval()
    val_losses = []

    with torch.no_grad():
        for batch in val_loader:
            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = model(**batch)
            val_loss = outputs.loss

            if not torch.isfinite(val_loss):
                raise RuntimeError(
                    f"Nonfinite validation loss in epoch {epoch}"
                )

            val_losses.append(
                float(val_loss.detach().cpu())
            )

    mean_val_loss = float(np.mean(val_losses))

    training_history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": mean_val_loss,
        }
    )

    print()
    print(
        f"Epoch {epoch} complete | "
        f"train loss={train_loss:.4f} | "
        f"validation loss={mean_val_loss:.4f}"
    )
    print()

training_time = time.time() - training_start
final_training_loss = training_history[-1]["train_loss"]
final_validation_loss = training_history[-1]["validation_loss"]

print("Training complete.")
print(f"Final training loss: {final_training_loss:.6f}")
print(f"Final validation loss: {final_validation_loss:.6f}")
print(f"Training time: {training_time:.2f} seconds")


epoch=1 step=1/500 loss=2.6258 grad_norm=0.8221 lr=0.00000800
epoch=1 step=25/500 loss=2.1595 grad_norm=1.1973 lr=0.00020000
epoch=1 step=50/500 loss=1.8032 grad_norm=0.9461 lr=0.00018947
epoch=1 step=75/500 loss=1.8335 grad_norm=1.1361 lr=0.00017895
epoch=1 step=100/500 loss=1.6078 grad_norm=0.9051 lr=0.00016842
epoch=1 step=125/500 loss=1.4934 grad_norm=1.1211 lr=0.00015789
epoch=1 step=150/500 loss=2.2153 grad_norm=0.9490 lr=0.00014737
epoch=1 step=175/500 loss=1.7406 grad_norm=1.1631 lr=0.00013684
epoch=1 step=200/500 loss=1.6159 grad_norm=1.1860 lr=0.00012632
epoch=1 step=225/500 loss=1.5928 grad_norm=0.9462 lr=0.00011579
epoch=1 step=250/500 loss=1.5374 grad_norm=0.9445 lr=0.00010526

Epoch 1 complete | train loss=1.8265 | validation loss=1.4534

epoch=2 step=275/500 loss=1.7865 grad_norm=1.0434 lr=0.00009474
epoch=2 step=300/500 loss=1.4760 grad_norm=1.1638 lr=0.00008421
epoch=2 step=325/500 loss=1.0186 grad_norm=1.0458 lr=0.00007368
epoch=2 step=350/500 loss=1.4689 grad_norm=0.

In [17]:
history_df = pd.DataFrame(training_history)
history_df


,epoch,train_loss,validation_loss
0,1,1.826498,1.453372
1,2,1.624805,1.439975


In [18]:
adapter_dir = "./saved_adapters/dialogsum_r4"
os.makedirs(adapter_dir, exist_ok=True)

model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

print("Saved LoRA adapter to:", adapter_dir)


Saved LoRA adapter to: ./saved_adapters/dialogsum_r4


## Task 5: Post Fine Tuning Inference

In [19]:
model.config.use_cache = True

r4_outputs = [
    generate_summary(model, example["dialogue"])
    for example in evaluation_examples
]

for i in range(2):
    print("=" * 80)
    print("EXAMPLE", i + 1)
    print()
    print("BEFORE FINE TUNING")
    print(baseline_outputs[i])
    print()
    print("AFTER FINE TUNING, LoRA r=4")
    print(r4_outputs[i])
    print()
    print("REFERENCE")
    print(evaluation_examples[i]["reference"])
    print()

assert all(x.strip() for x in r4_outputs)
print("Post training generation sanity check passed.")


EXAMPLE 1

BEFORE FINE TUNING
#Person2#: Thank you, sir.

AFTER FINE TUNING, LoRA r=4
#Person1# is ready to take a dictation for Ms. Dawson.

REFERENCE
Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.

EXAMPLE 2

BEFORE FINE TUNING
#Person2#: Thank you, sir.

AFTER FINE TUNING, LoRA r=4
#Person1# is ready to take a dictation for Ms. Dawson.

REFERENCE
In order to prevent employees from wasting time on Instant Message programs, #Person1# decides to terminate the use of those programs and asks Ms. Dawson to send out a memo to all employees by the afternoon.

Post training generation sanity check passed.


## Initial Output Comparison

In [20]:
comparison_df = pd.DataFrame(
    {
        "Example": [1, 2],
        "Reference": [
            example["reference"]
            for example in evaluation_examples
        ],
        "Baseline": baseline_outputs,
        "LoRA r=4": r4_outputs,
    }
)

pd.set_option("display.max_colwidth", None)
comparison_df


,Example,Reference,Baseline,LoRA r=4
0,1,Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.,"#Person2#: Thank you, sir.",#Person1# is ready to take a dictation for Ms. Dawson.
1,2,"In order to prevent employees from wasting time on Instant Message programs, #Person1# decides to terminate the use of those programs and asks Ms. Dawson to send out a memo to all employees by the afternoon.","#Person2#: Thank you, sir.",#Person1# is ready to take a dictation for Ms. Dawson.


## Initial Interpretation

This is only the first LoRA configuration. The required rank 16 experiment will be added in the next experimental iteration, so no final claim about the best LoRA rank is made here.


In [21]:
for i in range(2):
    print("=" * 80)
    print("Example", i + 1)
    print("Baseline:", baseline_outputs[i])
    print("LoRA r=4:", r4_outputs[i])
    print("Reference:", evaluation_examples[i]["reference"])
    print()

print(
    "The rank 4 adapter completed training with finite training and "
    "validation losses. The examples above provide the initial qualitative "
    "comparison against the pretrained baseline. Rank 16 will be tested "
    "separately before drawing a conclusion about LoRA rank."
)


Example 1
Baseline: #Person2#: Thank you, sir.
LoRA r=4: #Person1# is ready to take a dictation for Ms. Dawson.
Reference: Ms. Dawson helps #Person1# to write a memo to inform every employee that they have to change the communication method and should not use Instant Messaging anymore.

Example 2
Baseline: #Person2#: Thank you, sir.
LoRA r=4: #Person1# is ready to take a dictation for Ms. Dawson.
Reference: In order to prevent employees from wasting time on Instant Message programs, #Person1# decides to terminate the use of those programs and asks Ms. Dawson to send out a memo to all employees by the afternoon.

The rank 4 adapter completed training with finite training and validation losses. The examples above provide the initial qualitative comparison against the pretrained baseline. Rank 16 will be tested separately before drawing a conclusion about LoRA rank.


## Save Run Artifacts

In [22]:
comparison_df.to_csv(
    "outputs/initial_r4_comparison.csv",
    index=False,
)

history_df.to_csv(
    "outputs/r4_training_history.csv",
    index=False,
)

all_outputs = [
    {
        "example": i + 1,
        "dialogue": evaluation_examples[i]["dialogue"],
        "reference": evaluation_examples[i]["reference"],
        "baseline": baseline_outputs[i],
        "lora_r4": r4_outputs[i],
    }
    for i in range(2)
]

with open(
    "outputs/initial_r4_results.json",
    "w",
    encoding="utf8",
) as f:
    json.dump(
        all_outputs,
        f,
        indent=2,
        ensure_ascii=False,
    )

run_summary = {
    "timestamp": datetime.now().isoformat(),
    "SID4": SID4,
    "SEED": SEED,
    "SLICE": SLICE,
    "HP_ID": HP_ID,
    "CLS_A": CLS_A,
    "CLS_B": CLS_B,
    "model": MODEL_NAME,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "total_parameters": total_params,
    "trainable_parameters": trainable_params,
    "training_examples": len(train_dataset),
    "validation_examples": len(val_dataset),
    "epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "final_training_loss": final_training_loss,
    "final_validation_loss": final_validation_loss,
    "training_time_seconds": training_time,
}

with open(
    "outputs/r4_run_metrics.json",
    "w",
    encoding="utf8",
) as f:
    json.dump(run_summary, f, indent=2)

print(json.dumps(run_summary, indent=2))


{
  "timestamp": "2026-09-29T18:52:41.343810",
  "SID4": 2046,
  "SEED": 2046,
  "SLICE": 46,
  "HP_ID": 0,
  "CLS_A": 6,
  "CLS_B": 3,
  "model": "google/flan-t5-small",
  "lora_rank": 4,
  "lora_alpha": 16,
  "lora_dropout": 0.05,
  "total_parameters": 77133184,
  "trainable_parameters": 172032,
  "training_examples": 1000,
  "validation_examples": 100,
  "epochs": 2,
  "batch_size": 4,
  "learning_rate": 0.0002,
  "final_training_loss": 1.6248047552108764,
  "final_validation_loss": 1.4399751663208007,
  "training_time_seconds": 76.91101694107056
}


In [23]:
run_log = f'''DialogSum LoRA Rank 4 Run

Timestamp: {datetime.now().isoformat()}

SID4: {SID4}
SEED: {SEED}
SLICE: {SLICE}
HP_ID: {HP_ID}
CLS_A: {CLS_A}
CLS_B: {CLS_B}

Base model: {MODEL_NAME}
Device: {device}

Training examples: {len(train_dataset)}
Validation examples: {len(val_dataset)}

LoRA rank: {LORA_RANK}
LoRA alpha: {LORA_ALPHA}
LoRA dropout: {LORA_DROPOUT}
Trainable parameters: {trainable_params}
Total parameters: {total_params}

Epochs: {NUM_EPOCHS}
Batch size: {BATCH_SIZE}
Learning rate: {LEARNING_RATE}

Final training loss: {final_training_loss:.6f}
Final validation loss: {final_validation_loss:.6f}
Training time seconds: {training_time:.2f}

Training history:
{history_df.to_string(index=False)}
'''

with open("RUN_LOG.txt", "w", encoding="utf8") as f:
    f.write(run_log)

print(run_log)


DialogSum LoRA Rank 4 Run

Timestamp: 2026-09-29T18:52:41.351985

SID4: 2046
SEED: 2046
SLICE: 46
HP_ID: 0
CLS_A: 6
CLS_B: 3

Base model: google/flan-t5-small
Device: cuda

Training examples: 1000
Validation examples: 100

LoRA rank: 4
LoRA alpha: 16
LoRA dropout: 0.05
Trainable parameters: 172032
Total parameters: 77133184

Epochs: 2
Batch size: 4
Learning rate: 0.0002

Final training loss: 1.624805
Final validation loss: 1.439975
Training time seconds: 76.91

Training history:
 epoch  train_loss  validation_loss
     1    1.826498         1.453372
     2    1.624805         1.439975

